In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
!git clone https://github.com/votaquangnhat/music-timbre-transfer.git

Cloning into 'music-timbre-transfer'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 86 (delta 30), reused 78 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 237.02 KiB | 1.32 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [3]:
import torch

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No CUDA-enabled GPU detected.")

CUDA version: 12.8
GPU count: 1
Current GPU: 0
GPU name: Tesla T4


In [4]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [5]:
from pathlib import Path
from huggingface_hub import snapshot_download

PROJECT_DIR = Path("/content/music-timbre-transfer")  # change this if your repo folder has another name
%cd {PROJECT_DIR}

DATA_DIR = PROJECT_DIR / "data"
SPECTRORGRAMS_DIR = DATA_DIR / "spectrogram"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAR_PATH = Path("/content/drive/MyDrive/project2/spectrogram.rar")

# Mount Google Drive if not mounted
from google.colab import drive
drive.mount("/content/drive")

# Install unrar
!apt-get -qq install -y unrar

# Check file exists
assert RAR_PATH.exists(), f"File not found: {RAR_PATH}"

# Extract spectrogram.rar into DATA_DIR
!unrar x -o+ "{RAR_PATH}" "{DATA_DIR}/"

print("Dataset extracted to:", DATA_DIR)
!find "{DATA_DIR}" -maxdepth 2 -type f | head

/content/music-timbre-transfer
Mounted at /content/drive

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/project2/spectrogram.rar

Creating    /content/music-timbre-transfer/data/spectrogram           OK
Extracting  /content/music-timbre-transfer/data/spectrogram/spectrogram_725_slice_47_flute.npy       0%  OK 
Extracting  /content/music-timbre-transfer/data/spectrogram/spectrogram_144_slice_9_guitar.npy       0%  OK 
Extracting  /content/music-timbre-transfer/data/spectrogram/spectrogram_812_slice_44_piano.npy       0%  OK 
Extracting  /content/music-timbre-transfer/data/spectrogram/spectrogram_534_slice_31_piano.npy       0%  OK 
Extracting  /content/music-timbre-transfer/data/spectrogram/spectrogram_16_slice_43_flute.npy       0%  OK 
Extracting  /content/music-timbre-transfer/data/spectrogram/spectrogram_582_slice_32_guitar.npy       0%  OK 
Extracting  /content/

In [6]:
!rm -rf {PROJECT_DIR}/pretrained

In [7]:

PRETRAINED_PATH = Path("/content/drive/MyDrive/project2/pretrained.rar")
PRETRAINED_DIR = PROJECT_DIR / "pretrained"

# Check file exists
assert PRETRAINED_PATH.exists(), f"File not found: {PRETRAINED_PATH}"

# Extract spectrogram.rar into DATA_DIR
!unrar x -o+ "{PRETRAINED_PATH}" "{PROJECT_DIR}/"


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/project2/pretrained.rar

Creating    /content/music-timbre-transfer/pretrained                 OK
Extracting  /content/music-timbre-transfer/pretrained/AE_real_instruments.pt       0%  1%  2%  3%  4%  OK 
Extracting  /content/music-timbre-transfer/pretrained/AE_slakh.pt          4%  5%  6%  7%  8%  OK 
Creating    /content/music-timbre-transfer/pretrained/real_instruments  OK
Extracting  /content/music-timbre-transfer/pretrained/real_instruments/checkpoint.pt       8%  9% 10% 11% 12% 13% 14% 15% 16% 17% 18% 19% 20% 21% 22% 23% 24% 25% 26% 27% 28% 29% 30% 31% 32% 33% 34% 35% 36% 37% 38%  OK 
Extracting  /content/music-timbre-transfer/pretrained/real_instruments/config.gin      38%  OK 
Cre

In [9]:
!pip install einops_exts torch_ema nnAudio tensorboard cached-conv cqt_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.8 MB/s eta 0:00:00


In [10]:
import sys
sys.path.append('./src')

SOURCE_TIMBRE = "piano"
TARGET_TIMBRE = "flute"
string_path = f"/content/drive/MyDrive/riffusion_lora_checkpoints/test_lora_{TARGET_TIMBRE}"
LORA_PATH = Path(string_path) / "checkpoint-5000"
!python src/evaluation.py \
  --mode external \
  --spectrograms_dir data/spectrogram \
  --subset test \
  --source_timbre {SOURCE_TIMBRE} \
  --target_timbre {TARGET_TIMBRE} \
  --external_target_subset test \
  --external_target_index 0 \
  --lora_path {LORA_PATH} \
  --external_nb_steps 40 \
  --external_guidance 2.0 \
  --output_dir outputs/eval_external_{SOURCE_TIMBRE}_to_{TARGET_TIMBRE} \
  --strength 0.3 \
  --guidance_scale 2.0 \
  --num_inference_steps 50 \
  --max_samples 80 \
  --save_examples \
  --num_examples_to_save 5

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Audio processing utilities loaded.
Loading base pipeline...
model_index.json: 100% 541/541 [00:00<00:00, 3.09MB/s]
Fetching 13 files: 100% 13/13 [00:30<00:00,  2.37s/it]
Download complete: 100% 4.27G/4.27G [00:30<00:00, 120MB/s]                
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]
Loading pipeline components...:  17% 1/6 [00:00<00:00,  5.45it/s]An error occurred while trying to fetch /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6e

In [11]:
import sys
sys.path.append('./src')

SOURCE_TIMBRE = "guitar"
TARGET_TIMBRE = "flute"
string_path = f"/content/drive/MyDrive/riffusion_lora_checkpoints/test_lora_{TARGET_TIMBRE}"
LORA_PATH = Path(string_path) / "checkpoint-5000"
!python src/evaluation.py \
  --mode external \
  --spectrograms_dir data/spectrogram \
  --subset test \
  --source_timbre {SOURCE_TIMBRE} \
  --target_timbre {TARGET_TIMBRE} \
  --external_target_subset test \
  --external_target_index 0 \
  --lora_path {LORA_PATH} \
  --external_nb_steps 40 \
  --external_guidance 2.0 \
  --output_dir outputs/eval_external_{SOURCE_TIMBRE}_to_{TARGET_TIMBRE} \
  --strength 0.3 \
  --guidance_scale 2.0 \
  --num_inference_steps 50 \
  --max_samples 80 \
  --save_examples \
  --num_examples_to_save 5

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Audio processing utilities loaded.
Loading base pipeline...
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]An error occurred while trying to fetch /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  33% 2/6 [00:13<00:21,  5.42s/it]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights:   1% 

In [12]:
import sys
sys.path.append('./src')

SOURCE_TIMBRE = "piano"
TARGET_TIMBRE = "guitar"
string_path = f"/content/drive/MyDrive/riffusion_lora_checkpoints/test_lora_{TARGET_TIMBRE}"
LORA_PATH = Path(string_path) / "checkpoint-5000"
!python src/evaluation.py \
  --mode external \
  --spectrograms_dir data/spectrogram \
  --subset test \
  --source_timbre {SOURCE_TIMBRE} \
  --target_timbre {TARGET_TIMBRE} \
  --external_target_subset test \
  --external_target_index 0 \
  --lora_path {LORA_PATH} \
  --external_nb_steps 40 \
  --external_guidance 2.0 \
  --output_dir outputs/eval_external_{SOURCE_TIMBRE}_to_{TARGET_TIMBRE} \
  --strength 0.3 \
  --guidance_scale 2.0 \
  --num_inference_steps 50 \
  --max_samples 80 \
  --save_examples \
  --num_examples_to_save 5

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Audio processing utilities loaded.
Loading base pipeline...
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights: 100% 196/196 [00:00<00:00, 1323.47it/s]
Loading pipeline components...:  50% 3/6 [00:00<00:00,  4.95it/s]An error occurred while trying to fetch /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/unet.
Defaulting to unsafe serialization. Pass `allow_pickle

In [13]:
import sys
sys.path.append('./src')

SOURCE_TIMBRE = "flute"
TARGET_TIMBRE = "guitar"
string_path = f"/content/drive/MyDrive/riffusion_lora_checkpoints/test_lora_{TARGET_TIMBRE}"
LORA_PATH = Path(string_path) / "checkpoint-5000"
!python src/evaluation.py \
  --mode external \
  --spectrograms_dir data/spectrogram \
  --subset test \
  --source_timbre {SOURCE_TIMBRE} \
  --target_timbre {TARGET_TIMBRE} \
  --external_target_subset test \
  --external_target_index 0 \
  --lora_path {LORA_PATH} \
  --external_nb_steps 40 \
  --external_guidance 2.0 \
  --output_dir outputs/eval_external_{SOURCE_TIMBRE}_to_{TARGET_TIMBRE} \
  --strength 0.3 \
  --guidance_scale 2.0 \
  --num_inference_steps 50 \
  --max_samples 80 \
  --save_examples \
  --num_examples_to_save 5

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Audio processing utilities loaded.
Loading base pipeline...
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights: 100% 196/196 [00:00<00:00, 1136.60it/s]
Loading pipeline components...:  17% 1/6 [00:00<00:02,  1.83it/s]An error occurred while trying to fetch /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=F

In [14]:
import sys
sys.path.append('./src')

SOURCE_TIMBRE = "flute"
TARGET_TIMBRE = "piano"
string_path = f"/content/drive/MyDrive/riffusion_lora_checkpoints/test_lora_{TARGET_TIMBRE}"
LORA_PATH = Path(string_path) / "checkpoint-5000"
!python src/evaluation.py \
  --mode external \
  --spectrograms_dir data/spectrogram \
  --subset test \
  --source_timbre {SOURCE_TIMBRE} \
  --target_timbre {TARGET_TIMBRE} \
  --external_target_subset test \
  --external_target_index 0 \
  --lora_path {LORA_PATH} \
  --external_nb_steps 40 \
  --external_guidance 2.0 \
  --output_dir outputs/eval_external_{SOURCE_TIMBRE}_to_{TARGET_TIMBRE} \
  --strength 0.3 \
  --guidance_scale 2.0 \
  --num_inference_steps 50 \
  --max_samples 80 \
  --save_examples \
  --num_examples_to_save 5

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Audio processing utilities loaded.
Loading base pipeline...
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]An error occurred while trying to fetch /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--riffusion--riffusion-model-v1/snapshots/8f2e752c74e8316c6eb4fdaa6598a46ce1d88af5/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  33% 2/6 [00:00<00:00,  5.58it/s]An error occurred while trying to fetch /root/.cache/huggingface/hub/m

In [15]:
import sys
sys.path.append('./src')

SOURCE_TIMBRE = "guitar"
TARGET_TIMBRE = "piano"
string_path = f"/content/drive/MyDrive/riffusion_lora_checkpoints/test_lora_{TARGET_TIMBRE}"
LORA_PATH = Path(string_path) / "checkpoint-5000"
!python src/evaluation.py \
  --mode external \
  --spectrograms_dir data/spectrogram \
  --subset test \
  --source_timbre {SOURCE_TIMBRE} \
  --target_timbre {TARGET_TIMBRE} \
  --external_target_subset test \
  --external_target_index 0 \
  --lora_path {LORA_PATH} \
  --external_nb_steps 40 \
  --external_guidance 2.0 \
  --output_dir outputs/eval_external_{SOURCE_TIMBRE}_to_{TARGET_TIMBRE} \
  --strength 0.3 \
  --guidance_scale 2.0 \
  --num_inference_steps 50 \
  --max_samples 80 \
  --save_examples \
  --num_examples_to_save 5

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Audio processing utilities loaded.
Loading base pipeline...
Loading pipeline components...:   0% 0/6 [00:00<?, ?it/s]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights:   1% 2/196 [00:00<00:55,  3.50it/s]
Loading weights:  29% 56/196 [00:00<00:01, 106.78it/s]
Loading weights:  42% 83/196 [00:00<00:00, 129.66it/s]
Loading weights:  55% 107/196 [00:01<00:00, 132.65it/s]
Loading weights:  65% 127/196 [00:01<00:00, 136.24it/s]
Loading weights:  74% 145/196 [00:01<00:00, 144.67it/s]
Loading weights:  83% 163/196 [00:01<00:00, 149.21it/s]
Loading weights: 100% 196/196 [00:01<00:00, 125.51it/s]
Loading pipeline components...:  17% 1/6 [00:02<00:10,  2.12s/it]An error occurred while trying 